# AlterNet 2.0 - g:Profiler Input (GTEx)

## Workflow

1. Load curated edges from Part 3 (plausibility-filtered)
2. Compute target scores (weighted in-degree)
3. Project transcript targets to genes (MAX policy)
4. Select top-K genes (K=500 default, configurable)
5. Export gene lists for g:Profiler web interface

## Setup and Configuration

In [31]:
import pandas as pd
import numpy as np
import os
import os.path as op
import yaml
from collections import defaultdict

In [32]:
CONDITION = "Kidney - Cortex"

# Paths
PART3_RESULTS = "./results_gtex_plau_filtering/"
PART2_RESULTS = "./results_gtex_edge_cate/"
GPROFILER_RESULTS = "./results_gtex_gprofiler_input/"
BIOMART_PATH = "biomart.txt"

# ORA parameters
TOP_K = 500
PROJECTION_METHOD = 'max'
USE_ORDERED_QUERY = True
ENRICHMENT_PVAL = 0.05

os.makedirs(GPROFILER_RESULTS, exist_ok=True)

Condition: Kidney - Cortex
Top K targets: 500
Ordered query: True
Output directory: ./results_gprofiler/


## Load Data

In [33]:
# Load curated tables from Part 3

set_a = pd.read_csv(op.join(PART3_RESULTS, f"{CONDITION}_set_a_plausible.tsv"), sep='\t')
set_b = pd.read_csv(op.join(PART3_RESULTS, f"{CONDITION}_set_b_plausible.tsv"), sep='\t')


Loading plausibility-filtered edge tables...
  Table 1 (TF tx→gene): 29,294 edges
  Table 2 (TF tx→tx): 28,223 edges


In [34]:
# Load filtered networks from Part 2

net1 = pd.read_csv(op.join(PART2_RESULTS, f"{CONDITION}_net1_filtered.tsv"), sep='\t')
net2 = pd.read_csv(op.join(PART2_RESULTS, f"{CONDITION}_net2_filtered.tsv"), sep='\t')
net3 = pd.read_csv(op.join(PART2_RESULTS, f"{CONDITION}_net3_filtered.tsv"), sep='\t')



Loading filtered networks...
  Net1 (gene→gene): 22,843 edges
  Net2 (tx→gene): 17,405 edges
  Net3 (tx→tx): 29,293 edges


In [35]:
# Load BioMart mappings

biomart = pd.read_csv(BIOMART_PATH, sep='\t')
tx2gene = dict(zip(biomart['Transcript stable ID'], biomart['Gene stable ID']))
gene2symbol = dict(zip(biomart['Gene stable ID'], biomart['Gene name']))
symbol2gene = dict(zip(biomart['Gene name'], biomart['Gene stable ID']))



Loading BioMart annotations...
  Transcripts mapped: 278,220
  Genes with symbols: 70,611


## Core Functions

In [ ]:
def score_targets(edges, target_col, importance_col='median_importance'):
    """
    Compute target scores as weighted in-degree.
    score(target) = sum of importance for all edges targeting it
    """
    scores = edges.groupby(target_col)[importance_col].sum()
    return scores.sort_values(ascending=False)

In [37]:
def project_tx_to_gene(tx_scores, tx2gene, method='max'):
    """
    Project transcript-level scores to gene-level.
    
    Parameters:
    - tx_scores: pd.Series mapping transcript_id -> score
    - tx2gene: dict mapping transcript_id -> gene_id
    - method: 'max' (default) or 'sum'
    
    Returns:
    - gene_scores: pd.Series mapping gene_id -> score
    - rep_tx: dict mapping gene_id -> representative transcript
    """
    df = pd.DataFrame({
        'transcript_id': tx_scores.index,
        'score': tx_scores.values
    })
    df['gene_id'] = df['transcript_id'].map(tx2gene)
    df = df.dropna(subset=['gene_id'])
    
    if method == 'max':
        idx = df.groupby('gene_id')['score'].idxmax()
        result = df.loc[idx].set_index('gene_id')
        gene_scores = result['score'].sort_values(ascending=False)
        rep_tx = result['transcript_id'].to_dict()
    elif method == 'sum':
        gene_scores = df.groupby('gene_id')['score'].sum().sort_values(ascending=False)
        rep_tx = {}
    else:
        raise ValueError(f"Unknown method: {method}")
    
    return gene_scores, rep_tx

In [ ]:
def build_target_list(edges, target_col, importance_col='median_importance',
                      target_type='gene', tx2gene=None, gene2symbol=None,
                      K=500, projection_method='max'):
    """
    Build a ranked target list from an edge set.
    
    Parameters:
    - edges: DataFrame with edges
    - target_col: column containing target IDs
    - importance_col: column containing importance scores
    - target_type: 'gene' or 'transcript'
    - tx2gene: transcript to gene mapping (required if target_type='transcript')
    - gene2symbol: gene ID to symbol mapping
    - K: number of top genes to return
    - projection_method: 'max' or 'sum' (for transcript targets)
    
    Returns:
    - top_df: DataFrame with ranked targets
    - gene_symbols: list of gene symbols for g:Profiler
    - metadata: dict with scoring info
    """
    if len(edges) == 0:
        return pd.DataFrame(), [], {'n_edges': 0}
    
    # Score targets
    target_scores = score_targets(edges, target_col, importance_col)
    
    # Project if transcript targets
    if target_type == 'transcript':
        if tx2gene is None:
            raise ValueError("tx2gene mapping required for transcript targets")
        gene_scores, rep_tx = project_tx_to_gene(target_scores, tx2gene, method=projection_method)
    else:
        gene_scores = target_scores
        rep_tx = {}
    
    # Get top K
    top_genes = gene_scores.head(K)
    
    # Build result DataFrame
    top_df = pd.DataFrame({
        'rank': range(1, len(top_genes) + 1),
        'gene_id': top_genes.index,
        'score': top_genes.values
    })
    
    if gene2symbol is not None:
        top_df['gene_symbol'] = top_df['gene_id'].map(gene2symbol)
        top_df = top_df.dropna(subset=['gene_symbol'])
        gene_symbols = top_df['gene_symbol'].tolist()
    else:
        gene_symbols = top_df['gene_id'].tolist()
    
    if len(rep_tx) > 0:
        top_df['rep_transcript'] = top_df['gene_id'].map(rep_tx)
    
    metadata = {
        'n_edges': len(edges),
        'n_targets': len(target_scores),
        'n_genes': len(gene_scores),
        'n_top_symbols': len(gene_symbols),
        'projection_method': projection_method if target_type == 'transcript' else 'none'
    }
    
    return top_df, gene_symbols, metadata

## Build the Three Target Lists

### L1: Net1 Canonical Baseline (Gene to Gene)

In [39]:
l1_top, l1_symbols, l1_meta = build_target_list(
    edges=net1,
    target_col='target_gene',
    importance_col='mean_importance',
    target_type='gene',
    gene2symbol=gene2symbol,
    K=TOP_K
)



Edges: 22,843
Unique targets: 6,884
Top 500 symbols: 500

Top 10 targets:
 rank         gene_id     score gene_symbol
    1 ENSG00000090054 17.645219      SPTLC1
    2 ENSG00000182774 17.206603       RPS17
    3 ENSG00000089009 16.832886        RPL6
    4 ENSG00000166441 16.765693      RPL27A
    5 ENSG00000198755 16.437127      RPL10A
    6 ENSG00000197858 16.371134       GPAA1
    7 ENSG00000254772 16.270150       EEF1G
    8 ENSG00000172809 16.173843       RPL38
    9 ENSG00000116138 15.932678     DNAJC16
   10 ENSG00000204628 15.807074       RACK1


### L2: Source Isoform-Specific (AlterNet 1.0 Style)

In [ ]:
# Filter Set A for source_isoform_specific edges
l2_edges = set_a[set_a['source_category'] == 'source_isoform_specific'].copy()

importance_col_l2 = 'S2_median'

l2_top, l2_symbols, l2_meta = build_target_list(
    edges=l2_edges,
    target_col='target_gene',
    importance_col=importance_col_l2,
    target_type='gene',
    gene2symbol=gene2symbol,
    K=TOP_K
)

### L3: Target Isoform-Unique (AlterNet 2.0)

In [41]:
# Filter Set B for target_isoform_unique edges
l3_table2 = set_b[set_b['target_category'] == 'target_isoform_specific'].copy()

# Get corresponding Net3 edges for transcript-level scoring
# Set B is at (regulator_tx, target_gene) level
# We need to link to Net3 which has (source_transcript, target_transcript)

# Strategy: Filter Net3 to TF-like edges and get transcript targets
net3_tf_like = net3[net3['reg_type'].isin(['TF', 'TF_SF'])].copy()

# Create lookup from Set B
l3_pairs = set(zip(l3_table2['regulator_tx'], l3_table2['target_gene']))

# Add target_gene to Net3 for matching
net3_tf_like['target_gene'] = net3_tf_like['target_transcript'].map(tx2gene)
net3_tf_like['pair'] = list(zip(net3_tf_like['source_transcript'], net3_tf_like['target_gene']))

# Filter to edges matching Set B target_isoform_unique
l3_edges = net3_tf_like[net3_tf_like['pair'].isin(l3_pairs)].copy()

# Build target list - targets are transcripts, project to genes
l3_top, l3_symbols, l3_meta = build_target_list(
    edges=l3_edges,
    target_col='target_transcript',
    importance_col='mean_importance',
    target_type='transcript',
    tx2gene=tx2gene,
    gene2symbol=gene2symbol,
    K=TOP_K,
    projection_method=PROJECTION_METHOD
)



Table 2 filtered rows: 12,034
Net3 matched edges: 14,101

Edges: 14,101
Unique transcript targets: 8,222
Projected to genes: 4,234
Top 500 symbols: 499

Top 10 targets:
 rank         gene_id     score gene_symbol  rep_transcript
    1 ENSG00000173253 13.584730       DMRT2 ENST00000302441
    2 ENSG00000186468 13.512307       RPS23 ENST00000504293
    3 ENSG00000138030 13.149664         KHK ENST00000469936
    4 ENSG00000100129 13.002305       EIF3L ENST00000482600
    5 ENSG00000125691 12.649392       RPL23 ENST00000394332
    6 ENSG00000132541 12.566031        RIDA ENST00000254878
    7 ENSG00000166840 12.558732     GLYATL1 ENST00000525608
    8 ENSG00000123358 12.545689       NR4A1 ENST00000394825
    9 ENSG00000250799 12.342702      PRODH2 ENST00000653904
   10 ENSG00000140263 12.274432        SORD ENST00000558580


## Compare Target Lists

In [42]:
# Compare overlap between lists
set_l1 = set(l1_symbols)
set_l2 = set(l2_symbols)
set_l3 = set(l3_symbols)

all_three = set_l1 & set_l2 & set_l3



List sizes:
  L1 (canonical): 500
  L2 (source_isoform_specific): 500
  L3 (target_isoform_unique): 499

Pairwise overlaps:
  L1 ∩ L2: 23 (4.6%)
  L1 ∩ L3: 143 (28.7%)
  L2 ∩ L3: 19 (3.8%)

Three-way overlap:
  L1 ∩ L2 ∩ L3: 1

Unique to each list:
  Only L1: 335
  Only L2: 459
  Only L3: 338


## Export Gene Lists for g:Profiler Web Interface

In [43]:
# Save target lists as TSV (with scores)
l1_top.to_csv(op.join(GPROFILER_RESULTS, f"{CONDITION}_L1_canonical.tsv"), sep='\t', index=False)
l2_top.to_csv(op.join(GPROFILER_RESULTS, f"{CONDITION}_L2_source_isoform.tsv"), sep='\t', index=False)
l3_top.to_csv(op.join(GPROFILER_RESULTS, f"{CONDITION}_L3_target_isoform.tsv"), sep='\t', index=False)

In [44]:
# Save simple gene lists for web paste (newline-separated)
def save_gene_list(symbols, filepath):
    """Save gene list as newline-separated text file."""
    with open(filepath, 'w') as f:
        f.write('\n'.join(symbols))
    return len(symbols)

n1 = save_gene_list(l1_symbols, op.join(GPROFILER_RESULTS, f"{CONDITION}_L1_genes.txt"))
n2 = save_gene_list(l2_symbols, op.join(GPROFILER_RESULTS, f"{CONDITION}_L2_genes.txt"))
n3 = save_gene_list(l3_symbols, op.join(GPROFILER_RESULTS, f"{CONDITION}_L3_genes.txt"))


  Kidney - Cortex_L1_genes.txt: 500 genes
  Kidney - Cortex_L2_genes.txt: 500 genes
  Kidney - Cortex_L3_genes.txt: 499 genes


In [45]:
# Create combined file for multi-query comparison
# Format: Each query on a new line, prefixed with ">"

combined_path = op.join(GPROFILER_RESULTS, f"{CONDITION}_all_lists_combined.txt")

with open(combined_path, 'w') as f:
    f.write(f">L1_Canonical\n")
    f.write(' '.join(l1_symbols) + '\n')
    f.write(f">L2_Source_Isoform\n")
    f.write(' '.join(l2_symbols) + '\n')
    f.write(f">L3_Target_Isoform\n")
    f.write(' '.join(l3_symbols) + '\n')



Saved combined multi-query file: ./results_gprofiler/Kidney - Cortex_all_lists_combined.txt

To use in g:Profiler web interface:
  1. Go to https://biit.cs.ut.ee/gprofiler/gost
  2. Paste the contents of each _genes.txt file
  3. For ordered query: Check 'Ordered query' option
  4. Compare results across the three lists
